## 3D Event Reconstruction

This notebook goes through the process of 3D recoil track reconstruction. Using the combination of camera and ITO readout

#### Contents

1. [Matching ITO and Camera Readout](###Matching-ITO-and-Camera-Readout)
2. [Camera Image Noise Addition](###Camera-Image-Noise-Addition)
3. [Subdividing Camera and ITO Images](###Subdividing-Camera-and-ITO-Images)
4. [Voxelisation](###Voxelisation)

### Matching ITO and Camera Readout

In [ ]:
# import and create file paths for ITO and camera images

import numpy as np
import matplotlib.pyplot as plt
import sys
import os

sys.path.append(os.path.abspath('../ANN-code'))
from data_methods import create_file_paths
import pandas as pd

ito_dirs = ["../ANN-code/Data/ito_npy0/C"]
cam_dirs = ["../ANN-code/Data/im0/C"]

ito_file_paths = create_file_paths(ito_dirs)
cam_file_paths = create_file_paths(cam_dirs)

print("Number of ITO images: ", len(ito_file_paths))
print("Number of camera images: ", len(cam_file_paths))

In [ ]:
# match up ito and cam file paths
import random

# Load the min_dim_CF4_true.csv file and extract the IDs
min_dim_df = pd.read_csv("../ANN-code/Data/min_dim_CF4_true.csv")
min_dim_list = list(min_dim_df.iloc[:, 0])
min_dim_keys = [os.path.basename(path).split('_')[-2] for path in min_dim_list]

cam_dict = {os.path.basename(path).split('_')[-2]: path for path in cam_file_paths}
# Remove min_dim_keys from cam_dict
initial_cam_dict_length = len(cam_dict)
for key in min_dim_keys:
    if key in cam_dict:
        del cam_dict[key]
removed_count = initial_cam_dict_length - len(cam_dict)
print("Number of keys removed from cam_dict: ", removed_count)

ito_dict = {os.path.basename(path).split('_')[-2]: path for path in ito_file_paths}

matched_paths = []
for key in cam_dict:
    if key in ito_dict:
        matched_paths.append([cam_dict[key], ito_dict[key]])

print("Number of matched paths: ", len(matched_paths))

# matched_paths = random.sample(matched_paths, 3000)
# print("Number of matched paths after sampling: ", len(matched_paths))

In [ ]:
from collections import defaultdict

# list of matched paths stratified by energy

# Function to extract energy from the file name
def extract_energy(file_path):
    file_name = os.path.basename(file_path)
    energy_str = file_name.split('_')[0]
    return float(energy_str.replace('keV', ''))

# Create a dictionary to hold lists of matched paths stratified by energy
matched_paths_energy = defaultdict(list)

# Populate the dictionary
for cam_path, ito_path in matched_paths:
    energy = extract_energy(cam_path)
    matched_paths_energy[energy].append((cam_path, ito_path))

# Convert defaultdict to a regular dict for easier handling
matched_paths_energy = dict(matched_paths_energy)

# order matched paths by energy
matched_paths_energy = dict(sorted(matched_paths_energy.items()))

# now create a sample of 100 that is evenly distributed across energy levels
# Create a new dictionary that contains only every 100th entry, maintaining order
sampled_paths_energy = {k: v for i, (k, v) in enumerate(matched_paths_energy.items()) if i % 30 == 0}

# Flatten the sampled paths from the new dictionary
sampled_paths = [list(path) for paths in sampled_paths_energy.values() for path in paths]

matched_paths = sampled_paths
print("Number of matched paths after sampling: ", len(matched_paths))


In [ ]:
# for each file path, create bb_event object
# associate both ito and camera image with it

from bb_event import Event3D

events = []

for cam_path, ito_path in matched_paths:
    cam_image = np.load(cam_path)
    ito_image = np.load(ito_path)
    events.append(Event3D(cam_path, cam_image, ito_path, ito_image))

In [ ]:
test_cam = events[200].cam_image
test_ito = events[200].ito_image

# plot raw cam and ito images

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(test_cam, cmap='viridis')
ax[0].set_title("Camera Image")
ax[1].imshow(test_ito, cmap='viridis')
ax[1].set_title("ITO Image")

### Preprocessing

In [ ]:
from image_preprocessing import noise_adder, gaussian_smoothing

dark_dir = "../ANN-code/Data/darks"
dark_list_number = 0 # CHANGE FOR EACH DIRECTORY/JOB
m_dark = np.load(f"{dark_dir}/master_dark_1x1.npy")
example_dark_list = np.load(
    f"{dark_dir}/quest_std_dark_{dark_list_number}.npy"
)

for event in events:
    event.cam_image = noise_adder(event.cam_image, m_dark, example_dark_list)

In [ ]:
# thresholding
from skimage.filters import threshold_otsu

for event in events:

    event.cam_image = gaussian_smoothing(event.cam_image, 3.5)
    
    cam_threshold = threshold_otsu(event.cam_image)

    cam_above_threshold = event.cam_image.copy()
    cam_above_threshold[cam_above_threshold < cam_threshold] = 0

    event.cam_image = cam_above_threshold


### Subdividing Camera and ITO Images

In [ ]:
def subdivxy(im,xscale,yscale):
    im_large = np.zeros((im.shape[0]*yscale, im.shape[1]*xscale))
    for i in np.arange(xscale):
        for j in np.arange(yscale):
            im_large[j::yscale,i::xscale] = im

    return im_large

for event in events:
    event.cam_image = subdivxy(event.cam_image, 2, 2)
    event.ito_image = subdivxy(event.ito_image, 42, 13)
    event.ito_image = gaussian_smoothing(event.ito_image, 14)

    ito_threshold = threshold_otsu(event.ito_image)
    ito_above_threshold = event.ito_image.copy()
    ito_above_threshold[ito_above_threshold < ito_threshold] = 0

    event.ito_image = ito_above_threshold


In [ ]:
# padding the images to match x-dimensions, symetrically from either side

for event in events:
    cam_x_pixels = event.cam_image.shape[1]
    ito_x_pixels = event.ito_image.shape[1]

    if cam_x_pixels > ito_x_pixels:
        pad_amount = (cam_x_pixels - ito_x_pixels) // 2
        event.ito_image = np.pad(event.ito_image, ((0, 0), (pad_amount, cam_x_pixels - ito_x_pixels - pad_amount)), mode='constant')
    elif ito_x_pixels > cam_x_pixels:
        pad_amount = (ito_x_pixels - cam_x_pixels) // 2
        event.cam_image = np.pad(event.cam_image, ((0, 0), (pad_amount, ito_x_pixels - cam_x_pixels - pad_amount)), mode='constant')

In [ ]:
# test plot

test_cam = events[200].cam_image
test_ito = events[200].ito_image
print(len(events))

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(test_cam, cmap='viridis')
ax[0].set_title("Camera Image")
ax[1].imshow(test_ito, cmap='viridis')
ax[1].set_title("ITO Image")
plt.show()

In [ ]:
# preprocessing check

for event in events:
    plt.imshow(event.cam_image, cmap='viridis')
    plt.title(event.cam_path.split('/')[-1])
    plt.show()
    
    plt.imshow(event.ito_image, cmap='viridis')
    plt.title(event.ito_path.split('/')[-1])
    plt.show()

In [ ]:
# test 3d plot


fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Plot test_cam on the xy plane
x_cam, y_cam = np.meshgrid(np.arange(test_cam.shape[1]), np.arange(test_cam.shape[0]))
ax.plot_surface(x_cam, y_cam, np.zeros_like(test_cam), rstride=1, cstride=1, facecolors=plt.cm.viridis(test_cam / np.max(test_cam)), shade=False)

# Plot test_ito on the xz plane
x_ito, z_ito = np.meshgrid(np.arange(test_ito.shape[1]), np.arange(test_ito.shape[0]))
ax.plot_surface(x_ito, np.zeros_like(test_ito), z_ito, rstride=1, cstride=1, facecolors=plt.cm.viridis(test_ito / np.max(test_ito)), shade=False)

# Plot a black yz surface with white text "No Readout"
y_no_readout, z_no_readout = np.meshgrid(np.arange(test_cam.shape[0]), np.arange(test_ito.shape[0]))
ax.plot_surface(np.zeros_like(y_no_readout), y_no_readout, z_no_readout, color='#440154', shade=False)

ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Z axis')

# Rotate the plot for better visibility
ax.view_init(elev=30, azim=45)

plt.show()

### Voxelisation

$$
𝑅_{i,𝑗,𝑘} = ITO_{i,𝑗} × \textsf{𝐼𝑚}_{𝑖,𝑘} 
$$
This is the equation to get voxels from our ITO and Image pixels.

In [ ]:
print("test cam shape: ", test_cam.shape)
print("test ito shape: ", test_ito.shape)

In [ ]:
# ensure cam and ito images match x-axis

print("test_cam shape:", test_cam.shape)
print("test_ito shape:", test_ito.shape)

assert test_cam.shape[1] == test_ito.shape[1]  # Ensure X-dim matches

# Compute the voxel matrix efficiently using broadcasting
R = np.einsum('ik,jk->kij', test_cam, test_ito)  # Optimized voxel matrix computation

print("Voxel matrix R shape:", R.shape)

nonzero_voxels = np.count_nonzero(R)
total_voxels = np.prod(R.shape)
print("Nonzero voxels:", nonzero_voxels)
print("Total voxels:", total_voxels) 
print("Sparsity:", 1 - (nonzero_voxels / total_voxels))

In [ ]:
from image_analysis import plot_voxels

plot_voxels(R)